In [29]:
import os
os.environ["MLFLOW_TRACKING_URI"]="https://dagshub.com/vinitaraorane17849/data-science-project.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"]="vinitaraorane17849"
os.environ["MLFLOW_TRACKING_PASSWORD"]="f5e59444dcc29a43e2657e8d0581f5da19752de3"
os.environ["MLFLOW_REGISTRY_URI"] = ""

In [19]:
import os
os.chdir('D:\DataScienceProject')
%pwd

'D:\\DataScienceProject'

In [ ]:
import mlflow
mlflow.autolog(disable=True)

print("Tracking URI:", mlflow.get_tracking_uri())
print("Registry URI:", mlflow.get_registry_uri())




Tracking URI: https://dagshub.com/vinitaraorane17849/data-science-project.mlflow
Registry URI: https://dagshub.com/vinitaraorane17849/data-science-project.mlflow


In [21]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    model_path: Path
    test_data_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str

In [22]:
from src.datascience.constants import *
from src.datascience.utils.common import read_yaml, create_directories, save_json

In [23]:
class ConfigManager:
    def __init__(self,
            config_filepath= CONFIG_FILE_PATH,
            params_filepath= PARAMS_FILE_PATH,
            schema_filepath= SCHEMA_FILE_PATH):
        self.config= read_yaml(config_filepath)
        self.params= read_yaml(params_filepath)
        self.schema= read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config=self.config.model_evaluation
        params=self.params.ElasticNet
        schema=self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config=ModelEvaluationConfig(
            root_dir= config.root_dir,
            model_path= config.model_path,
            test_data_path= config.test_data_path,
            all_params= params,
            metric_file_name= config.metric_file_name,
            target_column= schema.name,
            mlflow_uri= "https://dagshub.com/vinitaraorane17849/data-science-project.mlflow"

        )
        return model_evaluation_config

In [37]:
import os
from src.datascience import logger
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from urllib.parse import urlparse
import mlflow
import mlflow.sklearn
import numpy as np
import joblib
import pandas as pd
import tempfile


In [40]:
##component- Model Evaluation
class ModelEvaluation:
    def __init__(self,config:ModelEvaluationConfig):
        self.config=config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2
    
    """
    def log_into_mlflow(self):

        test_data = pd.read_csv(self.config.test_data_path)
        model= joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[[self.config.target_column]]

        #mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_typ_store = urlparse(mlflow.get_tracking_uri()).scheme

        with mlflow.start_run():

            predicted_qualities = model.predict(test_x)

            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

            #Saving metrics as local
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)

            mlflow.log_params(self.config.all_params)

            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)

            #Model registry does not work with file store
        
            
            if tracking_url_typ_store not in ["file"] and "dagshub" not in mlflow.get_tracking_uri():

                #register the model
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")

            else:
                mlflow.sklearn.log_model(model, "model")
"""

    def log_into_mlflow(self):
        mlflow.autolog(disable=True)

        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)
        test_y = test_data[self.config.target_column]

        mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

        with mlflow.start_run():
            preds = model.predict(test_x)
            rmse, mae, r2 = self.eval_metrics(test_y, preds)

            mlflow.log_params(self.config.all_params)
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("mae", mae)
            mlflow.log_metric("r2", r2)

            # ✅ SAFE WAY: manual artifact logging
            with tempfile.TemporaryDirectory() as tmpdir:
                model_path = os.path.join(tmpdir, "model.joblib")
                joblib.dump(model, model_path)
                mlflow.log_artifact(model_path, artifact_path="model")





In [42]:
import joblib
import mlflow

run_id = "beb92f027aa8476fb581c787562652a6"
model_path = mlflow.artifacts.download_artifacts(
    run_id=run_id,
    artifact_path="model/model.joblib"
)

model = joblib.load(model_path)


d:\DataScienceProject\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:

try:
    config= ConfigManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation = ModelEvaluation(config=model_evaluation_config)
    model_evaluation.log_into_mlflow()

except Exception as e:
    raise e

https://dagshub.com/vinitaraorane17849/data-science-project.mlflow
[2025-12-10 21:15:41,166: INFO: common: yaml file: config\config.yaml loaded successfully]
[2025-12-10 21:15:41,168: INFO: common: yaml file: params.yaml loaded successfully]
[2025-12-10 21:15:41,168: INFO: common: yaml file: schema.yaml loaded successfully]
[2025-12-10 21:15:41,172: INFO: common: created directory at: artifacts]
[2025-12-10 21:15:41,175: INFO: common: created directory at: artifacts/model_evaluation]
🏃 View run funny-penguin-241 at: https://dagshub.com/vinitaraorane17849/data-science-project.mlflow/#/experiments/0/runs/beb92f027aa8476fb581c787562652a6
🧪 View experiment at: https://dagshub.com/vinitaraorane17849/data-science-project.mlflow/#/experiments/0
